<a href="https://colab.research.google.com/github/Zahraa28/K-Means-Model/blob/main/K_mean_clutering_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
vjchoudhary7_customer_segmentation_tutorial_in_python_path = kagglehub.dataset_download('vjchoudhary7/customer-segmentation-tutorial-in-python')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#import pandas as pd
#import numpy as np
from sklearn.model_selection import train_test_split
#from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
#from sklearn.linear_model import LinearRegression
#import os
import matplotlib.pyplot as plt
from sklearn import preprocessing
from sklearn.cluster import KMeans
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
path = '/kaggle/input/customer-segmentation-tutorial-in-python/Mall_Customers.csv'

df = pd.read_csv(os.path.join(path))
df

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated()

In [ ]:
label_encoder = LabelEncoder()

df['Gender'] = label_encoder.fit_transform(df['Gender'])
df

In [ ]:
df = df.set_index('CustomerID')
df

In [ ]:
corr_matrix = df.corr()

plt.figure(figsize=(10,10))
sns.heatmap(corr_matrix, cbar = True, square = True, fmt='.1f', annot=True, annot_kws={'size':8}, cmap='Blues')


In [ ]:


df.describe()

In [ ]:
df.info()

In [ ]:
df.count()

In [ ]:
df.columns

In [ ]:
df.nlargest(5,'Age')

In [ ]:
df.nsmallest(5,'Age')

In [ ]:
df.idxmax()

In [ ]:
df.shape

In [ ]:
df.sample(frac=0.5)

In [ ]:
df['Age'].value_counts()

In [ ]:
sns.scatterplot(data=df, x='Annual Income (k$)', y= 'Age', hue='Spending Score (1-100)')

plt.title("Income vs Spending Score by Age")
plt.show()

In [ ]:
# Split dataset
X = df[[ 'Annual Income (k$)', 'Age']]
y = df[['Spending Score (1-100)']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=0)

In [ ]:
# Normalize data
X_train_norm = preprocessing.normalize(X_train)
X_test_norm = preprocessing.normalize(X_test)

In [ ]:
kmeans = KMeans(n_clusters = 3, random_state = 0, n_init = 'auto')
kmeans.fit(X_train_norm)

In [ ]:
sns.scatterplot(data = X_train, x='Annual Income (k$)', y= 'Age', hue=kmeans.labels_)
plt.title("KMeans Clusters")
plt.legend(title="Cluster")
plt.show()

In [ ]:
sns.boxplot(x = kmeans.labels_, y = X_train['Annual Income (k$)'])
plt.title("Annual Income Distribution per Cluster")
plt.xlabel("Cluster")
plt.ylabel("Annual Income (k$)")
plt.show()

In [ ]:
# Silhouette Score
score_max = silhouette_score(X_train_norm, kmeans.labels_, metric='euclidean')
print(f"Silhouette Score: {score:.3f}")

In [ ]:
X_cluster = df[['Annual Income (k$)', 'Spending Score (1-100)']]
inertia = []
score = []

scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train_norm)   # shape (n_samples, n_features)

sil_scores_std = []

for k in range(2, 11):
    model = KMeans(n_clusters=k, random_state = 0, n_init='auto')
    model.fit(X_train_std)
    inertia.append(model.inertia_)
     # Append the silhouette score to scores
    score.append(silhouette_score(X_train_norm, model.labels_, metric='euclidean'))

score

In [ ]:
plt.plot(range(2, 11), score, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Scores for k')
plt.grid(True)
plt.show()

In [ ]:
# Plot the elbow curve
plt.plot(range(2, 11), inertia, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia (Within-cluster sum of squares)')
plt.title('Elbow Method for Optimal k')
plt.grid(True)
plt.show()

In [ ]:
sns.scatterplot(data = X_train, x='Annual Income (k$)', y= 'Age', hue = inertia.labels_)
plt.title("KMeans Clusters")
plt.legend(title="Cluster")
plt.show()

AttributeError: 'list' object has no attribute 'labels_'

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import seaborn as sns
import matplotlib.pyplot as plt

# Scale your input features
scaler = StandardScaler()
X_cluster_std = scaler.fit_transform(X_cluster)

inertia = []
silhouette_scores = []
models = []

# Try different values of k
for k in range(2, 11):
    model = KMeans(n_clusters=k, random_state=0, n_init='auto')
    model.fit(X_cluster_std)
    models.append(model)
    inertia.append(model.inertia_)
    silhouette_scores.append(silhouette_score(X_cluster_std, model.labels_))

# Find best model by silhouette score
best_k_index = silhouette_scores.index(max(silhouette_scores))
best_model = models[best_k_index]

# Add the best cluster labels to the original data (not standardized)
df['Cluster'] = best_model.labels_

# Plot the clusters
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', hue='Cluster', palette='Set2')
plt.title(f"KMeans Clusters (k={best_model.n_clusters}) - Best Silhouette Score")
plt.legend(title="Cluster")
plt.show()
